In [1]:
%reload_ext autoreload
%autoreload 2

import os
import sys

sys.path.append(".")
sys.path.append("..")

import jax
from jax import jit, vmap
import jax.numpy as jnp
import numpy as np
from jax.config import config
config.update("jax_enable_x64", True)

import numpyro.distributions as dist
from models.scd import dnds
from functools import partial

from models.psf import KingPSF

/n/home07/yitians/.conda/envs/torch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import numpy as np

## fake data

In [3]:
sys.path.append("../nptfit")
from nptfit_func import psf_corr

In [ ]:
kp = KingPSF()
f_ary, df_rho_div_f_ary = psf_corr(psf_r_func = kp.psf_fermi_r, num_f_bins=30)
f_ary = jnp.asarray(f_ary)
df_rho_div_f_ary = jnp.asarray(df_rho_div_f_ary)

In [8]:
npixROI = 100
theta = jnp.array([
    [0.01, 5.0, 1.3, -5.4, 11., 4.],
    [0.03, 5.5, 1.5, -5.5, 7.6, 2.],
])
pt_sum_compressed = jnp.asarray(np.random.uniform(size=(npixROI,)))
npt_compressed = jnp.asarray(np.random.uniform(size=(2, npixROI)))
data = jnp.asarray(np.round(10*np.random.uniform(size=(npixROI,))), dtype=jnp.int32)

k_max = int(np.max(data))

In [16]:
%timeit jnp.sum(log_like_np(theta, pt_sum_compressed, npt_compressed, data, f_ary, df_rho_div_f_ary, k_max, npixROI)).block_until_ready()

132 µs ± 2.25 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [17]:
#@partial(jit, static_argnums=(6,7,))
def log_like_np(theta, pt_sum_compressed, npt_compressed, data, f_ary, df_rho_div_f_ary, k_max, npixROI):
    """ Organize and combine non-Poissonian likelihoods across multiple templates
    """
    
    x_m_ary = jnp.zeros((npixROI, k_max + 1))
    x_m_sum = jnp.zeros(npixROI)

    s_ary = jnp.logspace(0., 2.5, 200)
    
    return_x_m_vmapped = vmap(return_x_m, in_axes=(None, None, 0, None, 0, None))
        
    dnds_ary = vmap(dnds, in_axes=(None,0))(s_ary, theta)
    x_m, x_m_sum = return_x_m_vmapped(f_ary, df_rho_div_f_ary, npt_compressed, s_ary, dnds_ary, k_max)
    print(x_m.shape, x_m_sum.shape)
    
    x_m = jnp.sum(x_m, axis=0)
    x_m_sum = jnp.sum(x_m_sum, axis=0)
    
    return log_like_internal(pt_sum_compressed, data, x_m, x_m_sum, k_max, npixROI)

@partial(jit, static_argnums=(4,5,))
def log_like_internal(pt_sum_compressed, data, x_m_ary, x_m_sum, k_max, npixROI):
    """ Non-Poissonian likelihood for single template, given x_m and x_m_sum
    """

    f0_ary = -(pt_sum_compressed + x_m_sum)
    f1_ary = pt_sum_compressed + x_m_ary[:, 1]

    pk_ary = jnp.zeros((npixROI, k_max + 1))

    pk_ary = pk_ary.at[:, 0].set(jnp.exp(f0_ary))
    pk_ary = pk_ary.at[:, 1].set(pk_ary[:, 0] * f1_ary)
    
    for k in np.arange(2, k_max + 1):
                
        n = jnp.arange(k - 1)
        pk_ary = pk_ary.at[:, k].set(jnp.sum((k - n) / k * x_m_ary[:, k - n] * pk_ary[:, n], axis=1) + f1_ary * pk_ary[:, k - 1] / k)

    pk_dat_ary = pk_ary[jnp.arange(npixROI), data]
        
    return jnp.log(pk_dat_ary)

@partial(jit, static_argnums=(5,))
def return_x_m(f_ary, df_rho_div_f_ary, npt_compressed, s_ary, dnds_ary, k_max):
    """ Dedicated calculation of x_m and x_m_sum
    """

    m_ary = jnp.arange(k_max + 1 , dtype=jnp.float64)
    gamma_ary = jnp.exp(jax.lax.lgamma(m_ary + 1))

    x_m_ary = df_rho_div_f_ary[:, None] * f_ary[:, None] * jnp.trapz(((dnds_ary * jnp.exp(-jnp.outer(f_ary, s_ary)))[:, :, None] * jax.lax.pow(jnp.outer(f_ary, s_ary)[:, :, None], m_ary[None, None, :]) / gamma_ary), s_ary, axis=1)
    x_m_ary = jnp.sum(x_m_ary, axis=0)

    x_m_ary = jnp.outer(npt_compressed, x_m_ary)

    x_m_sum_ary = jnp.sum((df_rho_div_f_ary * f_ary)[:, None] * jnp.trapz(dnds_ary, s_ary), axis=0)
    x_m_sum_ary = jnp.sum(x_m_sum_ary, axis=0)

    x_m_sum_ary = npt_compressed * x_m_sum_ary - x_m_ary[:, 0]

    return x_m_ary, x_m_sum_ary

In [18]:
jnp.sum(log_like_np(theta, pt_sum_compressed, npt_compressed, data, f_ary, df_rho_div_f_ary, k_max, npixROI))

(2, 100, 11) (2, 100)


Array(-308.09544556, dtype=float64)

In [25]:
#@partial(jit, static_argnums=(6,7,))
def log_like_np(theta, pt_sum_compressed, npt_compressed, data, f_ary, df_rho_div_f_ary, k_max, npixROI):
    """ Organize and combine non-Poissonian likelihoods across multiple templates
    """
    
    x_m_sum = jnp.zeros(npixROI)

    s_ary = jnp.logspace(0., 2.5, 200)
    
    return_x_m_vmapped = vmap(return_x_m, in_axes=(None, None, 0, None, 0, None))

    dnds_ary = vmap(dnds, in_axes=(None,0))(s_ary, theta)
    x_m_ary, x_m_sum_ary = return_x_m_vmapped(f_ary, df_rho_div_f_ary, npt_compressed, s_ary, dnds_ary, k_max)

    # new
    # x_m_ary (npt, kmax+1)
    # x_m_sum_ary (npt,)
    # npt_compressed (npt, npix)
    x_m = vmap(jnp.outer, in_axes=[0,0])(npt_compressed, x_m_ary) # (npt, npix, kmax+1)
    x_m_sum = npt_compressed * x_m_sum_ary[:,None] - x_m[:, :, 0] # (npt, npix)  = (npt, npix) * (npt, 1) - (npt, npix)
    # end new
    
    x_m = jnp.sum(x_m, axis=0) # (npix, kmax+1)
    x_m_sum = jnp.sum(x_m_sum, axis=0) # (npix,)
    
    return log_like_internal(pt_sum_compressed, data, x_m, x_m_sum, k_max, npixROI)

@partial(jit, static_argnums=(4,5,))
def log_like_internal(pt_sum_compressed, data, x_m_ary, x_m_sum, k_max, npixROI):
    """ Non-Poissonian likelihood for single template, given x_m and x_m_sum
    """

    f0_ary = -(pt_sum_compressed + x_m_sum)
    f1_ary = pt_sum_compressed + x_m_ary[:, 1]

    pk_ary = jnp.zeros((npixROI, k_max + 1))

    pk_ary = pk_ary.at[:, 0].set(jnp.exp(f0_ary))
    pk_ary = pk_ary.at[:, 1].set(pk_ary[:, 0] * f1_ary)
    
    for k in np.arange(2, k_max + 1):
                
        n = jnp.arange(k - 1)
        pk_ary = pk_ary.at[:, k].set(jnp.sum((k - n) / k * x_m_ary[:, k - n] * pk_ary[:, n], axis=1) + f1_ary * pk_ary[:, k - 1] / k)

    pk_dat_ary = pk_ary[jnp.arange(npixROI), data]
        
    return jnp.log(pk_dat_ary)

@partial(jit, static_argnums=(5,))
def return_x_m(f_ary, df_rho_div_f_ary, _, s_ary, dnds_ary, k_max):
    """ Dedicated calculation of x_m and x_m_sum
    """

    m_ary = jnp.arange(k_max + 1 , dtype=jnp.float64)
    gamma_ary = jnp.exp(jax.lax.lgamma(m_ary + 1))

    x_m_ary = df_rho_div_f_ary[:, None] * f_ary[:, None] * jnp.trapz(((dnds_ary * jnp.exp(-jnp.outer(f_ary, s_ary)))[:, :, None] * jax.lax.pow(jnp.outer(f_ary, s_ary)[:, :, None], m_ary[None, None, :]) / gamma_ary), s_ary, axis=1)
    x_m_ary = jnp.sum(x_m_ary, axis=0)

    x_m_sum_ary = jnp.sum((df_rho_div_f_ary * f_ary)[:, None] * jnp.trapz(dnds_ary, s_ary), axis=0)
    x_m_sum_ary = jnp.sum(x_m_sum_ary, axis=0)

    return x_m_ary, x_m_sum_ary

In [26]:
jnp.sum(log_like_np(theta, pt_sum_compressed, npt_compressed, data, f_ary, df_rho_div_f_ary, k_max, npixROI))

(2, 11) (2,)
(2, 100)


Array(-308.09544556, dtype=float64)